In [1]:
import gzip
import random
from pathlib import Path
from collections import Counter
import pyfastx
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

## Configuration

In [2]:
DATA_PATH = r'download.20260314.151350\Phytozome\PhytozomeV12\early_release\Ptrichocarpa_444_v3.1\assembly\Ptrichocarpa_444_v3.0.fa.gz'

## Data Loading and Inspection

In [3]:
sequences = pyfastx.Fasta(DATA_PATH)

lengths = [len(seq) for seq in sequences]

print('=== Sequence Length Stats ===')
print(f' Count : {len(lengths):,}')
print(f' Min : {min(lengths):,} bp')
print(f' Max : {max(lengths):,} bp')
print(f' Mean : {sum(lengths)/len(lengths):,.0f} bp')
print(f' Total : {sum(lengths)/1e6:.1f} Mb')

print('\n=== Sequence IDs (first 5) ===')
for seq in list(sequences)[:5]:
    print(f'  {seq.name:<20} {len(seq):>12,} bp')

=== Sequence Length Stats ===
 Count : 1,446
 Min : 1,003 bp
 Max : 50,495,391 bp
 Mean : 300,232 bp
 Total : 434.1 Mb

=== Sequence IDs (first 5) ===
  Chr01                  50,495,391 bp
  Chr02                  25,263,035 bp
  Chr03                  21,816,808 bp
  Chr04                  24,267,051 bp
  Chr05                  25,890,704 bp


## Tokenization
Overlaping k-mer tokenizer (stride = 1)

In [4]:
from itertools import product

k = 6
stride = 1

kmers   = [''.join(p) for p in product('ACGT', repeat=k)]
kmer_voc = {kmer: i for i, kmer in enumerate(kmers)}

print(f"Vocab size: {len(kmer_voc):,} possible {k}-mers")

all_token_ids = {}

for seq in sequences :

    dna_string = seq.seq.upper()

    ids = [
        kmer_voc.get(dna_string[i:i+k], -1) for i in range(0, len(dna_string) - k + 1, stride)
        ]

    all_token_ids[seq.name] = ids
    
    print(f"{seq.name:<30}  {len(seq):>12,} bp  →  {len(ids):>12,} tokens")
    print(f"first 4 k-mers : {[dna_string[i:i+k] for i in range(0, 4*stride, stride)]}")
    print(f"encoded : {ids[:4]}")
    print()

print(f"Done - {len(all_token_ids)} sequences tokenized")

Vocab size: 4,096 possible 6-mers
Chr01                             50,495,391 bp  →    50,495,386 tokens
first 4 k-mers : ['ACCCAA', 'CCCAAA', 'CCAAAC', 'CAAACC']
encoded : [336, 1344, 1281, 1029]

Chr02                             25,263,035 bp  →    25,263,030 tokens
first 4 k-mers : ['AAACCC', 'AACCCT', 'ACCCTA', 'CCCTAA']
encoded : [21, 87, 348, 1392]

Chr03                             21,816,808 bp  →    21,816,803 tokens
first 4 k-mers : ['CACCTA', 'ACCTAA', 'CCTAAA', 'CTAAAC']
encoded : [1116, 368, 1472, 1793]

Chr04                             24,267,051 bp  →    24,267,046 tokens
first 4 k-mers : ['TCACAT', 'CACATC', 'ACATCC', 'CATCCA']
encoded : [3347, 1101, 309, 1236]

Chr05                             25,890,704 bp  →    25,890,699 tokens
first 4 k-mers : ['CCCGAA', 'CCGAAA', 'CGAAAC', 'GAAACC']
encoded : [1376, 1408, 1537, 2053]

Chr06                             27,912,125 bp  →    27,912,120 tokens
first 4 k-mers : ['CTAACC', 'TAACCC', 'AACCCT', 'ACCCTA']
encoded : [179